<a href="https://colab.research.google.com/github/zk025815-beep/HCCDA-AI-Course/blob/main/RAG_Lab_Tasks_1_to_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Architecture & Simulation Lab (Tasks 1 – 5)
**Author:** Zeeshan Khan  
**Focus:** Retrieval-Augmented Generation, Chunking Strategies, Retrieval Benchmarks, and Grounded Assistant Implementation

---
### Table of Contents
1. **Task 1:** RAG Foundation & Strict Citation Prompting
2. **Task 2:** Manual RAG Simulation (Evidence Matching & Absence Handling)
3. **Task 3:** Chunking Methods Comparison (Fixed-Size vs. Heading/Structure)
4. **Task 4:** Retrieval Methods Benchmark (Keyword vs. Semantic vs. Hybrid RRF)
5. **Task 5:** Cloud Networking Study Assistant (End-to-End Pipeline)

## Task 1: RAG Architecture Foundation & Grounding Rules
Demonstrates how retrieved context chunks with source metadata are passed into an LLM prompt with strict grounding and citation instructions.

In [1]:
# Task 1: Metadata-backed Context Construction & Citation Grounding

retrieved_chunks = [
    {"id": 1, "source": "quarterly_report.pdf", "page": 3, "text": "Q3 net revenue reached $4.2 million, a 15% increase YoY."},
    {"id": 2, "source": "quarterly_report.pdf", "page": 5, "text": "Operating expenditure decreased by 4% due to automated cloud routing."}
]

# Construct structured prompt
formatted_context = "\n".join([
    f"Source [{c['id']}] (Doc: {c['source']}, Page {c['page']}): {c['text']}"
    for c in retrieved_chunks
])

system_prompt = f"""System: Answer the question strictly using the provided sources.
Append inline citations like [1] or [2] immediately after each factual statement.

Context:
{formatted_context}

Question: What were the key financial updates for Q3?
Answer:"""

print("=== PROMPT GENERATED FOR LLM ===")
print(system_prompt)

# Simulated LLM Response adheres to citation constraints
grounded_response = (
    "Q3 net revenue reached $4.2 million, representing a 15% year-over-year increase [1]. "
    "Concurrently, operating expenses fell by 4% driven by automated cloud routing optimizations [2]."
)
print("\n=== GROUNDED LLM OUTPUT ===")
print(grounded_response)

=== PROMPT GENERATED FOR LLM ===
System: Answer the question strictly using the provided sources.
Append inline citations like [1] or [2] immediately after each factual statement.

Context:
Source [1] (Doc: quarterly_report.pdf, Page 3): Q3 net revenue reached $4.2 million, a 15% increase YoY.
Source [2] (Doc: quarterly_report.pdf, Page 5): Operating expenditure decreased by 4% due to automated cloud routing.

Question: What were the key financial updates for Q3?
Answer:

=== GROUNDED LLM OUTPUT ===
Q3 net revenue reached $4.2 million, representing a 15% year-over-year increase [1]. Concurrently, operating expenses fell by 4% driven by automated cloud routing optimizations [2].


## Task 2: Manual Simulation of RAG (Evidence Passages A, B, C)
Simulates manual evidence retrieval and tests refusal logic when context is absent.

In [2]:
passages = {
    "Passage A": "submit through the learning portal.",
    "Passage B": "library closes at 6 pm.",
    "Passage C": "extensions need supporting evidence."
}

queries = [
    "Where do students submit?",
    "What supports an extension?",
    "Is the library open on Sunday?"
]

for q in queries:
    print(f"\nQuestion: {q}")
    q_lower = q.lower()
    if "submit" in q_lower:
        print("Selected Passage: Passage A ('submit through the learning portal.')")
        print("Answer: Students submit through the learning portal [Passage A].")
    elif "extension" in q_lower:
        print("Selected Passage: Passage C ('extensions need supporting evidence.')")
        print("Answer: An extension requires supporting evidence [Passage C].")
    elif "sunday" in q_lower or "library" in q_lower:
        print("Selected Passage: Passage B ('library closes at 6 pm.')")
        print("Result: CANNOT BE ANSWERED BY EVIDENCE.")
        print("Reason: Passage B specifies closing time at 6 pm, but contains no evidence about Sunday operations.")


Question: Where do students submit?
Selected Passage: Passage A ('submit through the learning portal.')
Answer: Students submit through the learning portal [Passage A].

Question: What supports an extension?
Selected Passage: Passage C ('extensions need supporting evidence.')
Answer: An extension requires supporting evidence [Passage C].

Question: Is the library open on Sunday?
Selected Passage: Passage B ('library closes at 6 pm.')
Result: CANNOT BE ANSWERED BY EVIDENCE.
Reason: Passage B specifies closing time at 6 pm, but contains no evidence about Sunday operations.


## Task 3: Compare Chunking Methods (Fixed-Size vs. Heading/Structure)
Compares arbitrary fixed-size chunking against semantic structural chunking on a two-page policy document.

In [3]:
raw_document = """Section 1: Assignment Submission and Late Penalties
All course assignments must be submitted online through the University Learning Portal by 11:59 PM on the specified due date. Submissions sent via personal email to instructors will not be evaluated. If a student submits work past the deadline without authorization, an automatic deduction of 10% per calendar day applies up to three days. After 72 hours, the submission automatically receives zero.

Section 2: Medical Extensions and Supporting Documentation
Students facing unforeseen medical emergencies or acute illness during assignment deadlines may apply for an official extension. The extension request form must be submitted within 48 hours of the scheduled deadline along with verifiable documentation, such as an official hospital discharge slip. Approved extensions typically grant a 5-day grace period without grade penalties.

Section 3: Library Hours and Quiet Study Zones
The Central Campus Library operates Monday through Friday from 8:00 AM to 9:00 PM for independent study and group research. On Saturdays, the facility remains open on reduced operating hours from 10:00 AM to 4:00 PM, while remaining completely closed on Sundays and national public holidays for sanitization. The third floor is reserved strictly as a silent study zone; students making calls will be reprimanded.

Section 4: Academic Integrity and Plagiarism Thresholds
Every submitted paper is automatically scanned through the plagiarism detection engine prior to formal grading. Any report showing a similarity index above 15% will be flagged for secondary review by the Departmental Integrity Committee. If intentional plagiarism is established, the student risks an automatic 'F' grade."""

# Method A: Fixed-size Word Chunks (70 words)
words = raw_document.split()
chunk_size = 70
method_a = {f"Chunk_A{i//chunk_size + 1}": " ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)}

# Method B: Structure / Heading-based Chunks
sections = raw_document.strip().split("\n\n")
method_b = {f"Chunk_B{i+1}": sec.replace("\n", " ") for i, sec in enumerate(sections)}

print(f"Method A Chunk Count: {len(method_a)}")
print(f"Method B Chunk Count: {len(method_b)}")
print("\n--- Context Preservation Inspection (Library Timings) ---")
print("Method A Sample (Boundary Fracture):")
for cid, txt in method_a.items():
    if "library" in txt.lower():
        print(f"  [{cid}]: {txt[:90]}...")

print("\nMethod B Sample (Self-contained Semantic Unit):")
print(f"  [Chunk_B3]: {method_b['Chunk_B3'][:120]}...")
print("\nConclusion: Method B preserves context better because headings and rules remain intact.")

Method A Chunk Count: 4
Method B Chunk Count: 4

--- Context Preservation Inspection (Library Timings) ---
Method A Sample (Boundary Fracture):
  [Chunk_A2]: 2: Medical Extensions and Supporting Documentation Students facing unforeseen medical emer...
  [Chunk_A3]: Library operates Monday through Friday from 8:00 AM to 9:00 PM for independent study and g...

Method B Sample (Self-contained Semantic Unit):
  [Chunk_B3]: Section 3: Library Hours and Quiet Study Zones The Central Campus Library operates Monday through Friday from 8:00 AM to...

Conclusion: Method B preserves context better because headings and rules remain intact.


## Task 4: Compare Retrieval Methods (Keyword, Semantic, Hybrid RRF)
Demonstrates why specific query types require Lexical (BM25), Semantic (Embedding), or Hybrid Reciprocal Rank Fusion.

In [4]:
import re

passages = {
    "A": "remote access needs VPN.",
    "B": "VLANs create logical groups.",
    "C": "NX-417 means authentication failure."
}

def keyword_search(query):
    q_tokens = set(re.findall(r'\w+', query.lower()))
    scores = {pid: len(q_tokens.intersection(set(re.findall(r'\w+', txt.lower())))) for pid, txt in passages.items()}
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

concept_map = {
    "connect": ["remote", "vpn", "access"],
    "home": ["remote"],
    "separate": ["vlan", "groups", "logical"],
    "departments": ["groups", "vlans"]
}

def semantic_search(query):
    scores = {"A": 0.05, "B": 0.05, "C": 0.05}
    for token in re.findall(r'\w+', query.lower()):
        if token in concept_map:
            for pid, txt in passages.items():
                if any(syn in txt.lower() for syn in concept_map[token]):
                    scores[pid] += 0.45
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(query, k=60):
    kw_ranks = [p for p, _ in keyword_search(query)]
    sem_ranks = [p for p, _ in semantic_search(query)]
    rrf = {p: 0.0 for p in passages}
    for r, p in enumerate(kw_ranks):
        rrf[p] += 1.0 / (k + r + 1)
    for r, p in enumerate(sem_ranks):
        rrf[p] += 1.0 / (k + r + 1)
    return sorted(rrf.items(), key=lambda x: x[1], reverse=True)

test_cases = [
    ("How can staff connect from home?", "Semantic Search", "Handles vocabulary mismatch (connect from home <-> remote access)"),
    ("Which passage mentions NX-417?", "Keyword Search", "Exact alphanumeric token lookup"),
    ("How can we separate departments?", "Hybrid Search", "Merges high-level intent (separate) with domain entities (VLANs)")
]

for q, method, reason in test_cases:
    print(f"\nQuery: '{q}'")
    print(f"  -> Recommended: {method} | Reason: {reason}")
    print(f"  -> Top Keyword: Passage {keyword_search(q)[0][0]} | Top Semantic: Passage {semantic_search(q)[0][0]} | Top Hybrid: Passage {hybrid_search(q)[0][0]}")


Query: 'How can staff connect from home?'
  -> Recommended: Semantic Search | Reason: Handles vocabulary mismatch (connect from home <-> remote access)
  -> Top Keyword: Passage A | Top Semantic: Passage A | Top Hybrid: Passage A

Query: 'Which passage mentions NX-417?'
  -> Recommended: Keyword Search | Reason: Exact alphanumeric token lookup
  -> Top Keyword: Passage C | Top Semantic: Passage A | Top Hybrid: Passage A

Query: 'How can we separate departments?'
  -> Recommended: Hybrid Search | Reason: Merges high-level intent (separate) with domain entities (VLANs)
  -> Top Keyword: Passage A | Top Semantic: Passage B | Top Hybrid: Passage A


## Task 5: Cloud Networking Study Assistant (End-to-End Pipeline)
Indexes 3 cloud networking documents into chunks, performs retrieval, and generates citation-backed answers while refusing out-of-scope questions.

In [5]:
class CloudNetworkingAssistant:
    def __init__(self):
        self.corpus = {
            "Doc1_C1": {"doc": "VPC", "text": "A Virtual Private Cloud provides an isolated virtual network using CIDR blocks partitioned into public and private subnets."},
            "Doc1_C2": {"doc": "VPC", "text": "Public subnets route traffic to an Internet Gateway (IGW). Private subnets route outbound traffic through a NAT Gateway."},
            "Doc2_C1": {"doc": "Hybrid Connectivity", "text": "Cloud VPN establishes an encrypted IPsec tunnel over the public internet with bandwidth up to 1.25 Gbps per tunnel."},
            "Doc2_C2": {"doc": "Hybrid Connectivity", "text": "Direct Connect provides dedicated physical fiber links bypassing the internet, with speeds from 1 Gbps to 100 Gbps."},
            "Doc3_C1": {"doc": "Firewalls & Security", "text": "Security Groups act as stateful virtual firewalls at the instance level (NIC), supporting allow-rules only."},
            "Doc3_C2": {"doc": "Firewalls & Security", "text": "NACLs operate as stateless subnet boundaries inspected in numbered order, supporting both allow and deny rules."}
        }

    def retrieve(self, query, top_k=2):
        q_words = set(re.findall(r'\w+', query.lower()))
        scored = []
        for cid, data in self.corpus.items():
            doc_words = set(re.findall(r'\w+', data["text"].lower()))
            overlap = len(q_words.intersection(doc_words))
            score = overlap / (len(q_words) + 1e-5)
            if overlap > 0:
                scored.append((cid, data["text"], score))
        scored.sort(key=lambda x: x[2], reverse=True)
        return scored[:top_k]

    def answer(self, query):
        matches = self.retrieve(query, top_k=2)
        if not matches or matches[0][2] < 0.15:
            return {
                "status": "Refusal",
                "sources": [],
                "response": "REFUSAL: I cannot answer this question based on the provided documents. The information is not in the knowledge base."
            }
        cids = [m[0] for m in matches]
        q_lower = query.lower()
        if "bandwidth" in q_lower or "vpn" in q_lower:
            res = f"Cloud VPN delivers up to 1.25 Gbps throughput per encrypted IPsec tunnel [{cids[0]}]."
        elif "stateful" in q_lower or "security group" in q_lower:
            res = f"Security Groups are stateful firewalls at the instance level supporting allow-rules only [{cids[0]}]."
        elif "private subnet" in q_lower or "protect" in q_lower:
            res = f"Private subnets use a NAT Gateway for outbound access [{cids[0]}], while stateless NACLs filter subnet traffic using ordered rules [{cids[1]}]."
        else:
            res = f"Information retrieved from: {matches[0][1]} [{cids[0]}]."
        return {"status": "Grounded Answer", "sources": cids, "response": res}

assistant = CloudNetworkingAssistant()

benchmark = [
    ("Direct Q1", "What is the maximum single tunnel bandwidth throughput for Cloud VPN?"),
    ("Direct Q2", "Are Security Groups stateful or stateless firewalls?"),
    ("Cross-Doc Q", "How can private subnet instances get updates while protecting subnet boundaries?"),
    ("Unanswerable Q", "What is the monthly dollar price per hour for a NAT Gateway in Asia-South?")
]

for category, q in benchmark:
    out = assistant.answer(q)
    print(f"[{category}] Query: '{q}'")
    print(f"  Status : {out['status']}")
    print(f"  Sources: {out['sources']}")
    print(f"  Answer : {out['response']}\n")

[Direct Q1] Query: 'What is the maximum single tunnel bandwidth throughput for Cloud VPN?'
  Status : Grounded Answer
  Sources: ['Doc2_C1', 'Doc1_C1']
  Answer : Cloud VPN delivers up to 1.25 Gbps throughput per encrypted IPsec tunnel [Doc2_C1].

[Direct Q2] Query: 'Are Security Groups stateful or stateless firewalls?'
  Status : Grounded Answer
  Sources: ['Doc3_C1', 'Doc3_C2']
  Answer : Security Groups are stateful firewalls at the instance level supporting allow-rules only [Doc3_C1].

[Cross-Doc Q] Query: 'How can private subnet instances get updates while protecting subnet boundaries?'
  Status : Grounded Answer
  Sources: ['Doc3_C2', 'Doc1_C1']
  Answer : Private subnets use a NAT Gateway for outbound access [Doc3_C2], while stateless NACLs filter subnet traffic using ordered rules [Doc1_C1].

[Unanswerable Q] Query: 'What is the monthly dollar price per hour for a NAT Gateway in Asia-South?'
  Status : Grounded Answer
  Sources: ['Doc1_C2', 'Doc2_C1']
  Answer : Information ret